# 00_audit_inputs_and_reference_data

**Role.** Gatekeeper audit of input files, class IDs, validation labels, district fields, and reproducibility metadata.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [ ]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os
import sys
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "00_audit_inputs_and_reference_data.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


# 00_InputAudit_AllInOne

Run this notebook before the four main Paper 1 notebooks.

Recommended order:

```text
GEE 01
→ GEE 02
→ 00_InputAudit_AllInOne.ipynb
→ 01_DataAudit_FeatureSelection_TopN_Sensitivity.ipynb
→ 02_ModelEvaluation_Table3_Table4_Fig4.ipynb
→ 03_RF_SHAP_Interpretation_Fig5_Fig6.ipynb
→ 04_AreaValidation_Fig8_SpatialBlockCV.ipynb
```

This audit notebook checks:

1. Required GEE CSV exports.
2. Train/validation `class_id` values.
3. FULL97 feature count.
4. Sensor prefixes: `S2_`, `S1_`, `L89_`, `DEM_`.
5. Validation prediction/confusion matrix inputs.
6. District-name matching and area consistency for Figure 8.

It writes outputs to:

```text
Supplementary/input_audit_report.csv
Supplementary/located_files.csv
Supplementary/audit_summary_metrics.csv
Supplementary/feature_summary.csv
Supplementary/class_distribution_train_validation.csv
Supplementary/district_matching.csv
Supplementary/top30_missing_zero_risk_features.csv
```

Rules:

```text
PASS  = OK
WARN  = can run, but review carefully
ERROR = stop and fix before analysis
```

If you want the notebook to continue after reporting errors, set:

```python
STOP_ON_ERROR = False
```


In [3]:

# =============================================================================
# 00: INPUT AUDIT ALL-IN-ONE
# Dak Lak Coffee Mapping Paper 1
# Purpose:
#   Check required CSV inputs before running Notebook 01–04.
#   This notebook DOES NOT train models, modify data, or draw figures.
#   It only reports PASS / WARN / ERROR and exports audit CSV files.
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import datetime

# =============================================================================
# 1. SETTINGS
# =============================================================================

STOP_ON_ERROR = False             # Set True to stop on first ERROR; False continues and shows all issues.
EXPECTED_CLASS_IDS = list(range(1, 11))
EXPECTED_FULL_FEATURE_COUNT = 97
EXPECTED_DISTRICT_COUNT = 15      # Pre-July 2025 Dak Lak administrative structure
COFFEE_CLASS_IDS = [1, 2, 3]

SEARCH_DIRS = [
    Path('.'),
    Path('data/raw'),
    Path('data/raw/data_DakLak_Statistics'),
    Path('data'),
    Path('GEE_Exports_R3000'),
    Path('data_DakLak_Statistics'),
    Path('Supplementary'),
    Path('Figures'),
    Path('Figures'),
    Path('/mnt/data'),
]

OUT_DIR = SUPPLEMENTARY_DIR

# Core GEE sample exports
TRAIN_FILE = 'Table_TrainSamples_FullFeatureSpace_2024.csv'
VAL_FILE   = 'Table_ValSamples_FullFeatureSpace_2024.csv'
LABEL_COL  = 'class_id'

# Final validation inputs for Table 4 / Figure 4
PREDICTION_FILE_CANDIDATES = [
    'Table_ValPredictions_RF_Final_2024.csv',
    'Table_ValPredictions_DakLak2024_corrTop25.csv',
    'Table_ValidationPredictions_DakLak2024_corrTop25.csv',
    'Table_ValSamples_WithPredictions_DakLak2024_corrTop25.csv',
]

PRED_COL_CANDIDATES = [
    'classification',
    'predicted',
    'prediction',
    'pred_class',
    'pred_class_id',
    'classified',
]

CONFUSION_MATRIX_FILE_CANDIDATES = [
    'Table_ConfusionMatrix_RowNorm_Long_DakLak2024_corrTop25.csv',
    'Table_ConfusionMatrix_Long_DakLak2024_corrTop25.csv',
    'confusion_matrix_counts.csv',
]

# Area-validation inputs for Figure 8
OFFICIAL_STATS_FILE = 'daklak_coffee_mapping_validation_2024.csv'
GEE_DISTRICT_AREA_FILE = 'GEE_DakLak_Coffee_Area_By_District_2024.csv'
GEE_CLASS_AREA_FILE = 'Table_AreaStatistics_DakLak2024.csv'

OFFICIAL_REQUIRED_COLS = [
    'district_name_en',
    'district_name',
    'planted_area_ha',
]
OFFICIAL_OPTIONAL_COLS = [
    'harvested_area_ha',
]

GEE_DISTRICT_REQUIRED_COLS = [
    'district_name_raw',
    'mapped_coffee_area_ha',
    'mapped_sun_coffee_area_ha',
    'mapped_intercrop_coffee_area_ha',
    'mapped_newly_planted_coffee_area_ha',
]

GEE_CLASS_REQUIRED_COLS = [
    'class_id',
    'class_name',
    'area_ha',
]

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

issues = []
located_files = []
summary_rows = []


def add_issue(level, item, message):
    """Append a PASS/WARN/ERROR audit message."""
    issues.append({
        'level': level,
        'item': item,
        'message': message,
    })


def add_summary(section, metric, value):
    summary_rows.append({
        'section': section,
        'metric': metric,
        'value': value,
    })


def find_file(filename, required=True, label=None):
    """Search a filename in common project folders."""
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            located_files.append({
                'file_role': label or filename,
                'filename': filename,
                'status': 'FOUND',
                'path': str(p),
            })
            return p

    located_files.append({
        'file_role': label or filename,
        'filename': filename,
        'status': 'MISSING',
        'path': '',
    })
    if required:
        add_issue('ERROR', label or filename, 'Missing required input file.')
    else:
        add_issue('WARN', label or filename, 'Optional input file not found.')
    return None


def find_first_existing(candidates, required=False, label='candidate input'):
    """Find the first existing file among several possible names."""
    for fn in candidates:
        for d in SEARCH_DIRS:
            p = d / fn
            if p.exists():
                located_files.append({
                    'file_role': label,
                    'filename': fn,
                    'status': 'FOUND',
                    'path': str(p),
                })
                return p

    located_files.append({
        'file_role': label,
        'filename': ' | '.join(candidates),
        'status': 'MISSING',
        'path': '',
    })
    if required:
        add_issue('ERROR', label, 'None of the candidate files was found.')
    else:
        add_issue('WARN', label, 'No candidate file found.')
    return None


def read_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig')


def audit_required_columns(df, table_name, required_cols):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        add_issue('ERROR', table_name, f'Missing required column(s): {missing}')
    else:
        add_issue('PASS', table_name, f'All required columns present: {required_cols}')


def numeric_feature_columns(df, label_col=LABEL_COL):
    drop_cols = {
        label_col,
        'system:index', '.geo',
        'lon', 'lat', 'longitude', 'latitude',
        'split', 'gridId', 'grid_id',
        'rnd', 'random',
    }
    drop_cols.update(c for c in df.columns if str(c).startswith('Unnamed'))
    return [
        c for c in df.columns
        if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])
    ]


def audit_class_ids(df, table_name, label_col=LABEL_COL, require_all=True):
    if label_col not in df.columns:
        add_issue('ERROR', table_name, f'Missing label column: {label_col}')
        return []

    y = pd.to_numeric(df[label_col], errors='coerce')
    if y.isna().any():
        add_issue('ERROR', table_name, f'{label_col} contains missing or non-numeric values.')

    classes = sorted(y.dropna().astype(int).unique().tolist())
    invalid = sorted(set(classes) - set(EXPECTED_CLASS_IDS))
    missing = sorted(set(EXPECTED_CLASS_IDS) - set(classes))

    if invalid:
        add_issue('ERROR', table_name, f'Invalid class IDs found: {invalid}. Expected only 1–10.')
    elif require_all and missing:
        add_issue('ERROR', table_name, f'Missing class IDs: {missing}. Expected all 10 classes.')
    else:
        add_issue('PASS', table_name, f'Class IDs OK: {classes}')

    return classes


def audit_feature_table(df, table_name):
    feats = numeric_feature_columns(df, LABEL_COL)
    n = len(feats)

    if n == EXPECTED_FULL_FEATURE_COUNT:
        add_issue('PASS', table_name, f'Feature count OK: {n}')
    else:
        add_issue('ERROR', table_name, f'Feature count = {n}; expected {EXPECTED_FULL_FEATURE_COUNT}.')

    group_counts = {
        'S2': sum(f.startswith('S2_') for f in feats),
        'S1': sum(f.startswith('S1_') for f in feats),
        'L89': sum(f.startswith('L89_') for f in feats),
        'DEM': sum(f.startswith('DEM_') for f in feats),
        'Other': sum(
            not (f.startswith('S2_') or f.startswith('S1_') or f.startswith('L89_') or f.startswith('DEM_'))
            for f in feats
        ),
    }

    if group_counts['Other'] > 0:
        add_issue('WARN', table_name, f"{group_counts['Other']} numeric feature(s) do not follow S2_/S1_/L89_/DEM_ prefixes.")
    else:
        add_issue('PASS', table_name, 'Feature prefixes OK: only S2_/S1_/L89_/DEM_ detected.')

    missing_rate = df[feats].isna().mean().sort_values(ascending=False) if feats else pd.Series(dtype=float)
    zero_rate = (df[feats] == 0).mean().sort_values(ascending=False) if feats else pd.Series(dtype=float)

    high_missing = missing_rate[missing_rate > 0.05]
    high_zero = zero_rate[zero_rate > 0.50]

    if len(high_missing):
        add_issue('WARN', table_name, f'{len(high_missing)} feature(s) have >5% missing values.')
    else:
        add_issue('PASS', table_name, 'No feature has >5% missing values.')

    if len(high_zero):
        add_issue('WARN', table_name, f'{len(high_zero)} feature(s) have >50% zero values. Check FILL_MISSING_PIXELS or cloud compositing.')
    else:
        add_issue('PASS', table_name, 'No feature has >50% zero values.')

    feature_summary = pd.DataFrame({
        'feature': feats,
        'sensor_group': [
            'S2' if f.startswith('S2_') else
            'S1' if f.startswith('S1_') else
            'L89' if f.startswith('L89_') else
            'DEM' if f.startswith('DEM_') else
            'Other'
            for f in feats
        ],
        'missing_rate': [float(missing_rate.get(f, np.nan)) for f in feats],
        'zero_rate': [float(zero_rate.get(f, np.nan)) for f in feats],
    })

    add_summary(table_name, 'n_rows', len(df))
    add_summary(table_name, 'n_features', n)
    for k, v in group_counts.items():
        add_summary(table_name, f'n_features_{k}', v)

    return feats, group_counts, feature_summary, missing_rate, zero_rate


def normalize_district_name(name):
    """Normalize Vietnamese district names for robust joining."""
    if pd.isna(name):
        return ''

    s = str(name).strip().lower()
    s = s.replace('đ', 'd')
    s = unicodedata.normalize('NFD', s)
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    s = re.sub(r'[^a-z0-9]+', '', s)

    aliases = {
        'thixabuonho': 'buonho',
        'txbuonho': 'buonho',
        'buonho': 'buonho',
        'tpbuonmathuot': 'buonmathuot',
        'thanhphobuonmathuot': 'buonmathuot',
        'buonmathuot': 'buonmathuot',
        'krongan': 'krongana',
        'krongana': 'krongana',
        'krongbong': 'krongbong',
        'krongbuk': 'krongbuk',
        'krongnang': 'krongnang',
        'krongpak': 'krongpak',
        'lak': 'lak',
        'lac': 'lak',
        'mdrak': 'mdrak',
        'mdrac': 'mdrak',
        'cumgar': 'cumgar',
        'cugar': 'cumgar',
        'eakar': 'eakar',
        'eahleo': 'eahleo',
        'easup': 'easup',
        'easoup': 'easup',
        'bondon': 'bondon',
        'buondon': 'bondon',
    }
    return aliases.get(s, s)


def save_csv(df, filename):
    path = OUT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

# =============================================================================
# 3. CORE SAMPLE TABLE AUDIT: required for Notebook 01, 02, 03, 04 SpatialBlockCV
# =============================================================================

print('=== 00 INPUT AUDIT ALL-IN-ONE ===')
print(f'Run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Output folder: {OUT_DIR.resolve()}')

train_path = find_file(TRAIN_FILE, required=True, label='training sample table')
val_path = find_file(VAL_FILE, required=True, label='validation sample table')

train = None
val = None
all_feature_summaries = []
class_dist_tables = []

if train_path is not None:
    train = read_csv(train_path)
    print(f'\nTraining samples: {train_path} | shape={train.shape}')
    audit_required_columns(train, 'training sample table', [LABEL_COL])
    audit_class_ids(train, 'training sample table', LABEL_COL, require_all=True)
    train_feats, train_groups, fs, train_missing, train_zero = audit_feature_table(train, 'training sample table')
    fs.insert(0, 'table', 'train')
    all_feature_summaries.append(fs)

    cd = train[LABEL_COL].value_counts().sort_index().rename_axis('class_id').reset_index(name='n') if LABEL_COL in train.columns else pd.DataFrame()
    cd.insert(0, 'table', 'train')
    class_dist_tables.append(cd)

if val_path is not None:
    val = read_csv(val_path)
    print(f'Validation samples: {val_path} | shape={val.shape}')
    audit_required_columns(val, 'validation sample table', [LABEL_COL])
    audit_class_ids(val, 'validation sample table', LABEL_COL, require_all=True)
    val_feats, val_groups, fs, val_missing, val_zero = audit_feature_table(val, 'validation sample table')
    fs.insert(0, 'table', 'validation')
    all_feature_summaries.append(fs)

    cd = val[LABEL_COL].value_counts().sort_index().rename_axis('class_id').reset_index(name='n') if LABEL_COL in val.columns else pd.DataFrame()
    cd.insert(0, 'table', 'validation')
    class_dist_tables.append(cd)

if train is not None and val is not None:
    if set(train_feats) == set(val_feats):
        add_issue('PASS', 'train-validation feature alignment', 'Train and validation feature columns are identical.')
    else:
        missing_in_val = sorted(set(train_feats) - set(val_feats))
        missing_in_train = sorted(set(val_feats) - set(train_feats))
        add_issue('ERROR', 'train-validation feature alignment', f'{len(missing_in_val)} train feature(s) missing in validation; {len(missing_in_train)} validation feature(s) missing in train.')

    add_summary('combined samples', 'n_rows', len(train) + len(val))

# =============================================================================
# 4. FINAL VALIDATION INPUT AUDIT: required for Notebook 02 Table 4 / Figure 4
# =============================================================================

pred_path = find_first_existing(PREDICTION_FILE_CANDIDATES, required=False, label='validation prediction file')
cm_path = find_first_existing(CONFUSION_MATRIX_FILE_CANDIDATES, required=False, label='confusion matrix file')

if pred_path is None and cm_path is None:
    add_issue('ERROR', 'Table 4 / Figure 4 input', 'Need either a validation prediction file or a confusion matrix CSV.')
else:
    add_issue('PASS', 'Table 4 / Figure 4 input', 'At least one prediction/confusion-matrix input is available.')

if pred_path is not None:
    pred = read_csv(pred_path)
    print(f'\nValidation prediction file: {pred_path} | shape={pred.shape}')

    if LABEL_COL not in pred.columns:
        add_issue('ERROR', 'validation prediction file', f'Missing true-label column: {LABEL_COL}')
    else:
        audit_class_ids(pred, 'validation prediction file true classes', LABEL_COL, require_all=True)

    pred_col = next((c for c in PRED_COL_CANDIDATES if c in pred.columns), None)
    if pred_col is None:
        add_issue('ERROR', 'validation prediction file', f'Missing prediction column. Tried: {PRED_COL_CANDIDATES}')
    else:
        yhat = pd.to_numeric(pred[pred_col], errors='coerce')
        pred_classes = sorted(yhat.dropna().astype(int).unique().tolist())
        invalid_pred = sorted(set(pred_classes) - set(EXPECTED_CLASS_IDS))
        if invalid_pred:
            add_issue('ERROR', 'validation prediction file', f'Invalid predicted class IDs: {invalid_pred}')
        else:
            add_issue('PASS', 'validation prediction file', f'Prediction column detected: {pred_col}; predicted classes: {pred_classes}')
        add_summary('validation prediction file', 'n_rows', len(pred))
        add_summary('validation prediction file', 'prediction_column', pred_col)

if cm_path is not None:
    cm = read_csv(cm_path)
    print(f'Confusion matrix file: {cm_path} | shape={cm.shape}')

    long_cols = set(cm.columns)
    has_long_format = (
        {'true_class_id', 'pred_class_id'}.issubset(long_cols)
        or {'true_class', 'pred_class'}.issubset(long_cols)
    )
    has_count = 'count' in cm.columns
    has_row_norm = ('row_normalized_percent' in cm.columns or 'row_normalized_proportion' in cm.columns)

    if has_long_format and (has_count or has_row_norm):
        add_issue('PASS', 'confusion matrix file', 'Long-format confusion matrix detected.')
    elif cm.shape[0] >= 10 and cm.shape[1] >= 10:
        add_issue('WARN', 'confusion matrix file', 'Possible square confusion matrix detected; please confirm row/column labels are class IDs 1–10.')
    else:
        add_issue('ERROR', 'confusion matrix file', 'Could not detect valid long-format or 10x10 square confusion matrix.')

# =============================================================================
# 5. AREA-VALIDATION AUDIT: required for Notebook 04 Figure 8
# =============================================================================

official_path = find_file(OFFICIAL_STATS_FILE, required=True, label='official district statistics')
gee_district_path = find_file(GEE_DISTRICT_AREA_FILE, required=True, label='GEE district coffee area')
gee_class_path = find_file(GEE_CLASS_AREA_FILE, required=True, label='GEE class area statistics')

official = None
gee_district = None
gee_class = None

if official_path is not None:
    official = read_csv(official_path)
    print(f'\nOfficial district statistics: {official_path} | shape={official.shape}')
    audit_required_columns(official, 'official district statistics', OFFICIAL_REQUIRED_COLS)
    for c in OFFICIAL_OPTIONAL_COLS:
        if c not in official.columns:
            add_issue('WARN', 'official district statistics', f'Optional column not found: {c}')

    for c in ['planted_area_ha', 'harvested_area_ha']:
        if c in official.columns:
            official[c] = pd.to_numeric(official[c], errors='coerce')
            if official[c].isna().any():
                add_issue('ERROR', 'official district statistics', f'{c} contains missing or non-numeric values.')
            elif (official[c] < 0).any():
                add_issue('ERROR', 'official district statistics', f'{c} contains negative values.')
            else:
                add_issue('PASS', 'official district statistics', f'{c} is numeric and non-negative.')

if gee_district_path is not None:
    gee_district = read_csv(gee_district_path)
    print(f'GEE district coffee area: {gee_district_path} | shape={gee_district.shape}')
    audit_required_columns(gee_district, 'GEE district coffee area', GEE_DISTRICT_REQUIRED_COLS)

    for c in GEE_DISTRICT_REQUIRED_COLS:
        if c != 'district_name_raw' and c in gee_district.columns:
            gee_district[c] = pd.to_numeric(gee_district[c], errors='coerce')
            if gee_district[c].isna().any():
                add_issue('ERROR', 'GEE district coffee area', f'{c} contains missing or non-numeric values.')
            elif (gee_district[c] < 0).any():
                add_issue('ERROR', 'GEE district coffee area', f'{c} contains negative values.')
            else:
                add_issue('PASS', 'GEE district coffee area', f'{c} is numeric and non-negative.')

    needed = [
        'mapped_coffee_area_ha',
        'mapped_sun_coffee_area_ha',
        'mapped_intercrop_coffee_area_ha',
        'mapped_newly_planted_coffee_area_ha',
    ]
    if all(c in gee_district.columns for c in needed):
        component_sum = (
            gee_district['mapped_sun_coffee_area_ha']
            + gee_district['mapped_intercrop_coffee_area_ha']
            + gee_district['mapped_newly_planted_coffee_area_ha']
        )
        diff = gee_district['mapped_coffee_area_ha'] - component_sum
        max_abs_diff = float(diff.abs().max())
        add_summary('GEE district coffee area', 'max_abs_subclass_sum_difference_ha', round(max_abs_diff, 3))
        if max_abs_diff > 1.0:
            add_issue('WARN', 'GEE district coffee area', f'Max difference between coffee total and subclass sum = {max_abs_diff:.2f} ha.')
        else:
            add_issue('PASS', 'GEE district coffee area', 'Coffee total matches sum of three coffee subclasses within 1 ha.')

if gee_class_path is not None:
    gee_class = read_csv(gee_class_path)
    print(f'GEE class area statistics: {gee_class_path} | shape={gee_class.shape}')
    audit_required_columns(gee_class, 'GEE class area statistics', GEE_CLASS_REQUIRED_COLS)

    if 'class_id' in gee_class.columns:
        audit_class_ids(gee_class, 'GEE class area statistics', 'class_id', require_all=True)

    for c in ['area_ha', 'area_pct']:
        if c in gee_class.columns:
            gee_class[c] = pd.to_numeric(gee_class[c], errors='coerce')
            if gee_class[c].isna().any():
                add_issue('ERROR', 'GEE class area statistics', f'{c} contains missing or non-numeric values.')
            elif (gee_class[c] < 0).any():
                add_issue('ERROR', 'GEE class area statistics', f'{c} contains negative values.')
            else:
                add_issue('PASS', 'GEE class area statistics', f'{c} is numeric and non-negative.')

# District matching
if official is not None and gee_district is not None and {'district_name_en'}.issubset(official.columns) and {'district_name_raw'}.issubset(gee_district.columns):
    official_tmp = official.copy()
    gee_district_tmp = gee_district.copy()
    official_tmp['district_key'] = official_tmp['district_name_en'].apply(normalize_district_name)
    gee_district_tmp['district_key'] = gee_district_tmp['district_name_raw'].apply(normalize_district_name)

    official_keys = set(official_tmp['district_key'])
    gee_keys = set(gee_district_tmp['district_key'])

    if len(official_keys) != EXPECTED_DISTRICT_COUNT:
        add_issue('WARN', 'official district statistics', f'Official district count = {len(official_keys)}; expected {EXPECTED_DISTRICT_COUNT}.')
    else:
        add_issue('PASS', 'official district statistics', f'District count OK: {EXPECTED_DISTRICT_COUNT}')

    if len(gee_keys) != EXPECTED_DISTRICT_COUNT:
        add_issue('WARN', 'GEE district coffee area', f'GEE district count = {len(gee_keys)}; expected {EXPECTED_DISTRICT_COUNT}.')
    else:
        add_issue('PASS', 'GEE district coffee area', f'District count OK: {EXPECTED_DISTRICT_COUNT}')

    missing_in_gee = sorted(official_keys - gee_keys)
    missing_in_official = sorted(gee_keys - official_keys)

    if missing_in_gee:
        add_issue('ERROR', 'district-name matching', f'Official districts missing in GEE output: {missing_in_gee}')
    if missing_in_official:
        add_issue('ERROR', 'district-name matching', f'GEE districts missing in official statistics: {missing_in_official}')
    if not missing_in_gee and not missing_in_official:
        add_issue('PASS', 'district-name matching', 'Official and GEE district keys match.')

    district_matching = official_tmp[['district_name_en', 'district_name', 'district_key']].merge(
        gee_district_tmp[['district_name_raw', 'district_key']],
        on='district_key',
        how='outer'
    ).sort_values('district_key')
    save_csv(district_matching, 'district_matching.csv')

    print('\nDistrict matching table:')
    display(district_matching)

# Coffee area consistency between class-level and district-level GEE area exports
if gee_class is not None and gee_district is not None:
    if {'class_id', 'area_ha'}.issubset(gee_class.columns) and 'mapped_coffee_area_ha' in gee_district.columns:
        coffee_area_by_class = float(gee_class.loc[gee_class['class_id'].isin(COFFEE_CLASS_IDS), 'area_ha'].sum())
        coffee_area_by_district = float(gee_district['mapped_coffee_area_ha'].sum())
        if coffee_area_by_class > 0:
            rel_diff = abs(coffee_area_by_district - coffee_area_by_class) / coffee_area_by_class * 100
            add_summary('coffee area consistency', 'class_area_coffee_ha', round(coffee_area_by_class, 3))
            add_summary('coffee area consistency', 'district_sum_coffee_ha', round(coffee_area_by_district, 3))
            add_summary('coffee area consistency', 'relative_difference_percent', round(rel_diff, 4))
            if rel_diff > 1.0:
                add_issue('WARN', 'coffee area consistency', f'District coffee-area sum differs from class-area coffee total by {rel_diff:.2f}%.')
            else:
                add_issue('PASS', 'coffee area consistency', f'Class-level and district-level coffee areas agree within 1%: {rel_diff:.3f}%.')

# =============================================================================
# 6. EXPORT AUDIT OUTPUTS
# =============================================================================

issues_df = pd.DataFrame(issues)
located_files_df = pd.DataFrame(located_files)
summary_df = pd.DataFrame(summary_rows)

if all_feature_summaries:
    feature_summary_df = pd.concat(all_feature_summaries, ignore_index=True)
else:
    feature_summary_df = pd.DataFrame()

if class_dist_tables:
    class_distribution_df = pd.concat(class_dist_tables, ignore_index=True)
else:
    class_distribution_df = pd.DataFrame()

save_csv(issues_df, 'input_audit_report.csv')
save_csv(located_files_df, 'located_files.csv')
save_csv(summary_df, 'audit_summary_metrics.csv')
if not feature_summary_df.empty:
    save_csv(feature_summary_df, 'feature_summary.csv')
if not class_distribution_df.empty:
    save_csv(class_distribution_df, 'class_distribution_train_validation.csv')

# Additional top-risk feature table
if not feature_summary_df.empty:
    risk_features = feature_summary_df.copy()
    risk_features['max_missing_or_zero_rate'] = risk_features[['missing_rate', 'zero_rate']].max(axis=1)
    risk_features = risk_features.sort_values('max_missing_or_zero_rate', ascending=False).head(30)
    save_csv(risk_features, 'top30_missing_zero_risk_features.csv')

# =============================================================================
# 7. HUMAN-READABLE SUMMARY
# =============================================================================

n_error = int((issues_df['level'] == 'ERROR').sum()) if not issues_df.empty else 0
n_warn = int((issues_df['level'] == 'WARN').sum()) if not issues_df.empty else 0
n_pass = int((issues_df['level'] == 'PASS').sum()) if not issues_df.empty else 0

print('\n' + '=' * 80)
print('INPUT AUDIT SUMMARY')
print('=' * 80)
print(f'PASS : {n_pass}')
print(f'WARN : {n_warn}')
print(f'ERROR: {n_error}')
print(f'Output files written to: {OUT_DIR.resolve()}')

print('\nLocated files:')
display(located_files_df)

print('\nAudit report:')
# Show ERROR/WARN first, then PASS
if not issues_df.empty:
    order = {'ERROR': 0, 'WARN': 1, 'PASS': 2}
    display(issues_df.sort_values(by='level', key=lambda s: s.map(order)).reset_index(drop=True))
else:
    print('No audit issues recorded.')

if not summary_df.empty:
    print('\nSummary metrics:')
    display(summary_df)

if not feature_summary_df.empty:
    print('\nFeature group summary:')
    display(
        feature_summary_df.groupby(['table', 'sensor_group'])
        .size()
        .rename('n_features')
        .reset_index()
    )

if n_error > 0:
    message = f'Input audit failed with {n_error} ERROR item(s). Fix these before running Notebook 01–04.'
    if STOP_ON_ERROR:
        raise ValueError(message)
    else:
        print('\n' + message)
else:
    print('\nAUDIT PASSED: no ERROR items. You can run Notebook 01–04. Read WARN items before final manuscript submission.')


=== 00 INPUT AUDIT ALL-IN-ONE ===
Run time: 2026-05-12 08:10:12
Output folder: D:\2024_PhD_Research\Chap2_Mapping\paper_mapping_workflow\results\supplementary

INPUT AUDIT SUMMARY
PASS : 0
WARN : 2
ERROR: 6
Output files written to: D:\2024_PhD_Research\Chap2_Mapping\paper_mapping_workflow\results\supplementary

Located files:


,file_role,filename,status,path
0,training sample table,Table_TrainSamples_FullFeatureSpace_2024.csv,MISSING,
1,validation sample table,Table_ValSamples_FullFeatureSpace_2024.csv,MISSING,
2,validation prediction file,Table_ValPredictions_RF_Final_2024.csv | Table...,MISSING,
3,confusion matrix file,Table_ConfusionMatrix_RowNorm_Long_DakLak2024_...,MISSING,
4,official district statistics,daklak_coffee_mapping_validation_2024.csv,MISSING,
5,GEE district coffee area,GEE_DakLak_Coffee_Area_By_District_2024.csv,MISSING,
6,GEE class area statistics,Table_AreaStatistics_DakLak2024.csv,MISSING,



Audit report:


,level,item,message
0,ERROR,training sample table,Missing required input file.
1,ERROR,validation sample table,Missing required input file.
2,ERROR,official district statistics,Missing required input file.
3,ERROR,Table 4 / Figure 4 input,Need either a validation prediction file or a ...
4,ERROR,GEE district coffee area,Missing required input file.
5,ERROR,GEE class area statistics,Missing required input file.
6,WARN,confusion matrix file,No candidate file found.
7,WARN,validation prediction file,No candidate file found.



Input audit failed with 6 ERROR item(s). Fix these before running Notebook 01–04.



## Output rule

This audit notebook writes only audit CSV files to `Supplementary/`. It does not create model tables or manuscript figures.


In [ ]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)
